In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: c:\Users\SrikanthRachakulla\anaconda3\envs\zs_ai\python.exe
Python version: 3.10.0 | packaged by conda-forge | (default, Nov 10 2021, 13:20:59) [MSC v.1916 64 bit (AMD64)]


In [2]:
import openai
import dotenv
import azure.identity

print("openai:", openai.__version__)
print("Required libraries imported successfully")

openai: 2.53.0
Required libraries imported successfully


In [3]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")

print("Endpoint configured:", bool(endpoint))
print("API key configured :", bool(api_key))
print("Model configured   :", bool(model))
print("Model deployment   :", model)

Endpoint configured: True
API key configured : True
Model configured   : True
Model deployment   : gpt-4.1-mini


In [4]:
from openai import OpenAI

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

print("Azure model client created successfully")

Azure model client created successfully


In [5]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "Explain prior authorization in healthcare in two sentences."
        }
    ]
)

print(response.choices[0].message.content)

Prior authorization in healthcare is a process where healthcare providers must obtain approval from a patient's insurance company before delivering certain treatments, medications, or services. This ensures that the proposed care is medically necessary and covered under the patient's insurance plan.


In [6]:
print(type(response))
print(response)


<class 'openai.types.chat.chat_completion.ChatCompletion'>
ChatCompletion(id='chatcmpl-EBNXoM8hbhQJ9lWKgsULwExJL6ZgK', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Prior authorization in healthcare is a process where healthcare providers must obtain approval from a patient's insurance company before delivering certain treatments, medications, or services. This ensures that the proposed care is medically necessary and covered under the patient's insurance plan.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=178

In [7]:
print("Prompt tokens     :", response.usage.prompt_tokens)
print("Completion tokens :", response.usage.completion_tokens)
print("Total tokens      :", response.usage.total_tokens)

Prompt tokens     : 16
Completion tokens : 46
Total tokens      : 62


In [8]:
print("Prompt filter results:")
print(response.prompt_filter_results)

print("\nResponse content filter results:")
print(response.choices[0].content_filter_results)

Prompt filter results:
[{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}]

Response content filter results:
{'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}


In [9]:
print(response.usage.latency_checkpoint)

{'engine_tbt_ms': 14, 'engine_ttft_ms': 39, 'engine_ttlt_ms': 670, 'pre_inference_ms': 290, 'service_tbt_ms': 14, 'service_ttft_ms': 522, 'service_ttlt_ms': 1151, 'total_duration_ms': 868, 'user_visible_ttft_ms': 233}


In [10]:
response_with_system = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You are a healthcare payer domain assistant. Explain concepts accurately and concisely for technology professionals."
        },
        {
            "role": "user",
            "content": "Explain prior authorization."
        }
    ]
)

print(response_with_system.choices[0].message.content)

Prior authorization (PA) is a utilization management process used by health insurance payers to determine if a prescribed medical service, procedure, or medication is medically necessary before it is provided. The provider must obtain approval from the payer by submitting clinical information supporting the need for the service. This helps control costs, reduce unnecessary care, and ensure treatments align with evidence-based guidelines. If the PA is denied, the patient may be responsible for the cost unless an appeal overturns the decision.


In [11]:
print("WITHOUT SYSTEM MESSAGE")
print("-" * 50)
print(response.choices[0].message.content)

print("\nWITH SYSTEM MESSAGE")
print("-" * 50)
print(response_with_system.choices[0].message.content)

WITHOUT SYSTEM MESSAGE
--------------------------------------------------
Prior authorization in healthcare is a process where healthcare providers must obtain approval from a patient's insurance company before delivering certain treatments, medications, or services. This ensures that the proposed care is medically necessary and covered under the patient's insurance plan.

WITH SYSTEM MESSAGE
--------------------------------------------------
Prior authorization (PA) is a utilization management process used by health insurance payers to determine if a prescribed medical service, procedure, or medication is medically necessary before it is provided. The provider must obtain approval from the payer by submitting clinical information supporting the need for the service. This helps control costs, reduce unnecessary care, and ensure treatments align with evidence-based guidelines. If the PA is denied, the patient may be responsible for the cost unless an appeal overturns the decision.


In [12]:
response_controlled = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer domain assistant.

Rules:
1. Answer for a technology professional.
2. Use healthcare payer terminology.
3. Keep the response to exactly 3 bullet points.
4. Do not exceed 80 words.
"""
        },
        {
            "role": "user",
            "content": "Explain prior authorization."
        }
    ]
)

print(response_controlled.choices[0].message.content)

- Prior authorization is a utilization management process requiring healthcare payers’ approval before certain services, medications, or procedures are covered.  
- It helps control costs and ensures clinical appropriateness based on evidence-based guidelines.  
- The process involves submitting clinical information for review to confirm medical necessity before payment authorization is granted.


In [13]:
print("Prompt tokens     :", response_controlled.usage.prompt_tokens)
print("Completion tokens :", response_controlled.usage.completion_tokens)
print("Total tokens      :", response_controlled.usage.total_tokens)

Prompt tokens     : 62
Completion tokens : 63
Total tokens      : 125


In [14]:
simple_tokens = response.usage.total_tokens
controlled_tokens = response_controlled.usage.total_tokens

increase = controlled_tokens - simple_tokens
increase_pct = (increase / simple_tokens) * 100

print("Simple prompt tokens     :", simple_tokens)
print("Controlled prompt tokens :", controlled_tokens)
print("Additional tokens        :", increase)
print(f"Increase                  : {increase_pct:.1f}%")

Simple prompt tokens     : 62
Controlled prompt tokens : 125
Additional tokens        : 63
Increase                  : 101.6%


In [15]:
few_shot_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You are a healthcare payer assistant. Follow the response style shown in the examples."
        },
        {
            "role": "user",
            "content": "What is a deductible?"
        },
        {
            "role": "assistant",
            "content": "Deductible: The amount a member pays out-of-pocket before the health plan begins paying for covered services."
        },
        {
            "role": "user",
            "content": "What is coinsurance?"
        },
        {
            "role": "assistant",
            "content": "Coinsurance: The percentage of an allowed healthcare cost that a member pays after meeting the deductible."
        },
        {
            "role": "user",
            "content": "What is prior authorization?"
        }
    ]
)

print(few_shot_response.choices[0].message.content)

Prior Authorization: A requirement that a healthcare provider obtains approval from the health plan before delivering certain services or medications to ensure they are covered.
